In [ ]:
import pandas as pd 
import numpy as np 
import os 
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch.nn.functional as F
import clip
import glob

import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as transforms

In [ ]:
def get_cosine_similarity(t1, t2):
    return F.cosine_similarity(t1, t2).item()

def return_image_set(df):
    return set(
        set(df['img1']).union(set(df['img2']))
    )

In [ ]:
cats = ["bottle", "bowl", "chair", "cup", "door", "spoon", "table", "window"]
dfs = [
        pd.read_csv(f"../../../data/PanelA/all2all_corr_bysubj_forRplots_{cat}.csv", header=None) for cat in cats
    ]

df = pd.concat(dfs, ignore_index=True)
df.columns = ["subj", "img1", "img2", "obj", "valid", "hist_corr"]

In [ ]:
print(df.shape)
print()
print(df.head())

In [ ]:
all_imgs = return_image_set(df)
print(len(all_imgs))

In [ ]:
# NOTE
# Please follow the setup instructions in each linked repository

def extract_clip_features(target_imgs):
    """
    https://github.com/openai/CLIP
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    model, preprocess = clip.load("ViT-B/32", device=device)

    folder_path = "/path/to/images" # NOTE images will be released on Databrary
    all_features = {}
    
    # for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):
    for img_name in target_imgs:
        img_path = os.path.join(folder_path, img_name)
        image = preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

        with torch.no_grad():
            all_features[img_name] = model.encode_image(image).cpu()

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

def extract_pe_features(target_imgs):
    """
    https://github.com/facebookresearch/perception_models/tree/main
    """

    device = "cuda:1" if torch.cuda.is_available() else "cpu"
    
    folder_path = "/path/to/images" # NOTE images will be released on Databrary

    model = pe.VisionTransformer.from_config("PE-Spatial-L14-448", pretrained=True)
    model = model.to(device)
    preprocess = transforms.get_image_transform(model.image_size)

    all_features = {}

    for img_name in target_imgs:
        img_path = os.path.join(folder_path, img_name)
        image = Image.open(img_path).convert("RGB")
        
        with torch.no_grad():
            inputs = preprocess(image).unsqueeze(0).to(device)
            image_features, *_ = model(inputs)
        
            all_features[img_name] = image_features.cpu()

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [ ]:
feats = extract_pe_features(all_imgs)

In [ ]:
# this requires a slight modification to extract the CLS token of each model before computing cos sim
print(feats['030FD_011375.jpg'].shape)

In [ ]:
pfx = "all2all_corr_bysubj" if False else "NULL"
for cat in cats:
    temp_df = pd.read_csv(f"../../../data/PanelA/{pfx}_forRplots_{cat}.csv", header=None)
    temp_df.columns = ["subj", "img1", "img2", "obj", "valid", "hist_corr"]
    temp_df["clip_cos_sim"] = temp_df.apply(
        # this requires a slight modification to extract the CLS token of each model before computing cos sim
        lambda row: get_cosine_similarity(feats[row["img1"]], feats[row["img2"]]),
        axis=1
    )
    temp_df.to_csv(f"../../../data/PanelA/deep_features/PE/PE-Spatial-L/{pfx}_forRplots_{cat}_withPE-Spatial-L.csv", header=False, index=False)
    del temp_df
